In [1]:
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd

In [2]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

from drGAT.metrics import get_parsed_df

In [3]:
def extract_method_data(filename):
    match = re.match(r"([A-Za-zA-Z0-9]+)_([a-z0-9]+)\.sqlite3", filename)
    return match.groups() if match else ("unknown", "unknown")

In [8]:
all_dfs = []

for file in glob.glob("*.sqlite3"):
    method, data = extract_method_data(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        # ✅ COMPLETE 状態の数を表示
        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )

        if df_valid.shape[0] == 0:
            continue

        # user_attrs から評価指標を取り出す
        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)
        parsed_df["n_complete"] = n_complete

        # ハイパーパラメータ列だけ抽出
        df_params = df_valid[
            [c for c in df_valid.columns if "params" in c]
        ].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        # method / data 列を追加
        parsed_df["method"] = method
        parsed_df["data"] = data

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 最良インデックス取得
    best_idxs = summary_df.groupby(["data", "method"])["AUPR_num"].idxmax()
    best_df = summary_df.loc[best_idxs].drop(columns=["AUPR_num"])

    # ✅ インデックス設定: method, data の順で MultiIndex
    best_df.set_index(
        [
            "data",
            "method",
        ],
        inplace=True,
    )

    # ✅ data をキーに並べ替え（index の level 1 をソート）
    best_df.sort_index(level="data", inplace=True)

    # 任意の順に並べたいカラム
    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

    # その順に並び替える（残りのカラムはそのまま後ろへ）
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    # インデックスの data レベルを並び替え
    desired_data_order = [
        "nci",
        "gdsc1",
        "gdsc2",
        "ctrp",
    ]  # 小文字なら小文字にそろえる！

    # sort_index ではなく reindex で順序を明示
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: desired_data_order.index(x[0]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ GAT_ctrp.sqlite3: 703 trials completed
✅ GAT_gdsc1.sqlite3: 653 trials completed
✅ GAT_gdsc2.sqlite3: 734 trials completed
✅ GAT_nci.sqlite3: 2394 trials completed
✅ GATv2_ctrp.sqlite3: 678 trials completed
✅ GATv2_gdsc1.sqlite3: 674 trials completed
✅ GATv2_gdsc2.sqlite3: 631 trials completed
✅ GATv2_nci.sqlite3: 2622 trials completed
✅ Transformer_ctrp.sqlite3: 520 trials completed
✅ Transformer_gdsc1.sqlite3: 469 trials completed
✅ Transformer_nci.sqlite3: 2478 trials completed
❌ Failed to parse Transformer_gdsc2_recovered.sqlite3: 'Record does not exist.'
❌ Failed to parse Broken_Transformer_gdsc2.sqlite3: 'Record does not exist.'


n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                2394  0.876 (± 0.007)  0.914 (± 0.008)   
      GATv2              2622  0.875 (± 0.006)  0.915 (± 0.014)   
      Transformer        2478  0.868 (± 0.009)  0.915 (± 0.011)   
gdsc1 GAT                 653   0.79 (± 0.007)   0.787 (± 0.01)   
      GATv2               674  0.784 (± 0.009)  0.773 (± 0.016)   
      Transformer         469  0.803 (± 0.004)  0.794 (± 0.012)   
gdsc2 GAT                 734  0.805 (± 0.008)   0.81 (± 0.011)   
      GATv2               631  0.815 (± 0.012)   0.82 (± 0.008)   
ctrp  GAT                 703  0.848 (± 0.004)   0.853 (± 0.01)   
      GATv2               678  0.848 (± 0.004)  0.855 (± 0.005)   
      Transformer         520  0.861 (± 0.006)  0.867 (± 0.004)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.826 (± 0.012)  0.868 (± 0.007)  0.948 (± 0.003)   
      GATv2        0.824 (± 0.017)  0.867 (± 0.006)  0.948 (± 0.003)   
      Transformer  0.807 (± 0.023)  0.857 (± 0.012)  0.945 (± 0.003)   
gdsc1 GAT           0.78 (± 0.014)  0.784 (± 0.006)  0.867 (± 0.007)   
      GATv2        0.791 (± 0.011)  0.781 (± 0.008)  0.866 (± 0.006)   
      Transformer  0.805 (± 0.009)  0.799 (± 0.004)  0.884 (± 0.002)   
gdsc2 GAT          0.817 (± 0.018)   0.813 (± 0.01)  0.885 (± 0.006)   
      GATv2        0.825 (± 0.027)  0.822 (± 0.015)   0.892 (± 0.01)   
ctrp  GAT          0.854 (± 0.009)  0.854 (± 0.004)  0.924 (± 0.002)   
      GATv2        0.852 (± 0.007)  0.853 (± 0.003)  0.927 (± 0.002)   
      Transformer  0.866 (± 0.014)  0.867 (± 0.006)  0.935 (± 0.003)   

                              AUPR     Balanced_ACC  params_T_max  \
data  method                                                        
nci   GAT          0.946 (± 0.004)  0.875 (± 0.007)           NaN   
      GATv2        0.947 (± 0.003)  0.875 (± 0.006)           NaN   
      Transformer  0.943 (± 0.004)   0.867 (± 0.01)           NaN   
gdsc1 GAT          0.869 (± 0.007)   0.79 (± 0.007)           NaN   
      GATv2        0.867 (± 0.006)  0.785 (± 0.009)           NaN   
      Transformer  0.885 (± 0.004)  0.803 (± 0.004)           NaN   
gdsc2 GAT          0.889 (± 0.013)  0.805 (± 0.008)           NaN   
      GATv2        0.895 (± 0.011)  0.815 (± 0.012)           NaN   
ctrp  GAT          0.933 (± 0.003)  0.847 (± 0.004)           NaN   
      GATv2        0.936 (± 0.003)  0.847 (± 0.004)           NaN   
      Transformer  0.942 (± 0.002)  0.861 (± 0.006)           NaN   

                  params_activation  ...  params_hidden1  params_hidden2  \
data  method                         ...                                   
nci   GAT                      relu  ...             259              96   
      GATv2                    relu  ...             395             236   
      Transformer              relu  ...             289             215   
gdsc1 GAT                      gelu  ...             325             236   
      GATv2                    gelu  ...             400              85   
      Transformer              gelu  ...             283             134   
gdsc2 GAT                      gelu  ...             496              72   
      GATv2                    relu  ...             377             103   
ctrp  GAT                      gelu  ...             398              83   
      GATv2                    gelu  ...             436              93   
      Transformer              gelu  ...             368             153   

                   params_hidden3  params_is_zero_pad  params_lr  \
data  method                                                       
nci   GAT                      91                True   0.000265   
      GATv2                    80                True   0.000376   
      Transformer              66                Tr

In [9]:
best_df

n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                2394  0.876 (± 0.007)  0.914 (± 0.008)   
      GATv2              2622  0.875 (± 0.006)  0.915 (± 0.014)   
      Transformer        2478  0.868 (± 0.009)  0.915 (± 0.011)   
gdsc1 GAT                 653   0.79 (± 0.007)   0.787 (± 0.01)   
      GATv2               674  0.784 (± 0.009)  0.773 (± 0.016)   
      Transformer         469  0.803 (± 0.004)  0.794 (± 0.012)   
gdsc2 GAT                 734  0.805 (± 0.008)   0.81 (± 0.011)   
      GATv2               631  0.815 (± 0.012)   0.82 (± 0.008)   
ctrp  GAT                 703  0.848 (± 0.004)   0.853 (± 0.01)   
      GATv2               678  0.848 (± 0.004)  0.855 (± 0.005)   
      Transformer         520  0.861 (± 0.006)  0.867 (± 0.004)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.826 (± 0.012)  0.868 (± 0.007)  0.948 (± 0.003)   
      GATv2        0.824 (± 0.017)  0.867 (± 0.006)  0.948 (± 0.003)   
      Transformer  0.807 (± 0.023)  0.857 (± 0.012)  0.945 (± 0.003)   
gdsc1 GAT           0.78 (± 0.014)  0.784 (± 0.006)  0.867 (± 0.007)   
      GATv2        0.791 (± 0.011)  0.781 (± 0.008)  0.866 (± 0.006)   
      Transformer  0.805 (± 0.009)  0.799 (± 0.004)  0.884 (± 0.002)   
gdsc2 GAT          0.817 (± 0.018)   0.813 (± 0.01)  0.885 (± 0.006)   
      GATv2        0.825 (± 0.027)  0.822 (± 0.015)   0.892 (± 0.01)   
ctrp  GAT          0.854 (± 0.009)  0.854 (± 0.004)  0.924 (± 0.002)   
      GATv2        0.852 (± 0.007)  0.853 (± 0.003)  0.927 (± 0.002)   
      Transformer  0.866 (± 0.014)  0.867 (± 0.006)  0.935 (± 0.003)   

                              AUPR     Balanced_ACC  params_T_max  \
data  method                                                        
nci   GAT          0.946 (± 0.004)  0.875 (± 0.007)           NaN   
      GATv2        0.947 (± 0.003)  0.875 (± 0.006)           NaN   
      Transformer  0.943 (± 0.004)   0.867 (± 0.01)           NaN   
gdsc1 GAT          0.869 (± 0.007)   0.79 (± 0.007)           NaN   
      GATv2        0.867 (± 0.006)  0.785 (± 0.009)           NaN   
      Transformer  0.885 (± 0.004)  0.803 (± 0.004)           NaN   
gdsc2 GAT          0.889 (± 0.013)  0.805 (± 0.008)           NaN   
      GATv2        0.895 (± 0.011)  0.815 (± 0.012)           NaN   
ctrp  GAT          0.933 (± 0.003)  0.847 (± 0.004)           NaN   
      GATv2        0.936 (± 0.003)  0.847 (± 0.004)           NaN   
      Transformer  0.942 (± 0.002)  0.861 (± 0.006)           NaN   

                  params_activation  ...  params_hidden1  params_hidden2  \
data  method                         ...                                   
nci   GAT                      relu  ...             259              96   
      GATv2                    relu  ...             395             236   
      Transformer              relu  ...             289             215   
gdsc1 GAT                      gelu  ...             325             236   
      GATv2                    gelu  ...             400              85   
      Transformer              gelu  ...             283             134   
gdsc2 GAT                      gelu  ...             496              72   
      GATv2                    relu  ...             377             103   
ctrp  GAT                      gelu  ...             398              83   
      GATv2                    gelu  ...             436              93   
      Transformer              gelu  ...             368             153   

                   params_hidden3  params_is_zero_pad  params_lr  \
data  method                                                       
nci   GAT                      91                True   0.000265   
      GATv2                    80                True   0.000376   
      Transformer              66                Tr

In [5]:
data='gdsc2'
method='Transformer'
study = optuna.load_study(study_name=method, storage="sqlite:///Transformer_gdsc2_recovered.sqlite3")
df_all = study.trials_dataframe()
df_valid = df_all.dropna(
    subset=["values_0", "values_1", "values_2", "values_3"]
)

# user_attrs から評価指標を取り出す
df_metrics = df_valid.iloc[:, 8:-2]
df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
parsed_df = get_parsed_df(df_metrics)
parsed_df["n_complete"] = n_complete

# ハイパーパラメータ列だけ抽出
df_params = df_valid[
    [c for c in df_valid.columns if "params" in c]
].reset_index(drop=True)
parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

# method / data 列を追加
parsed_df["method"] = method
parsed_df["data"] = data

In [6]:
parsed_df["AUPR_num"] = parsed_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

# 最良インデックス取得
best_idxs = parsed_df.groupby(["data", "method"])["AUPR_num"].idxmax()
best_df = parsed_df.loc[best_idxs].drop(columns=["AUPR_num"])

desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

# その順に並び替える（残りのカラムはそのまま後ろへ）
other_cols = [col for col in best_df.columns if col not in desired_order]
best_df = best_df[desired_order + other_cols]
best_df

,n_complete,ACC,Precision,Recall,F1,AUROC,AUPR,Balanced_ACC,method,data
338,2477,0.819 (± 0.012),0.815 (± 0.024),0.843 (± 0.02),0.828 (± 0.011),0.895 (± 0.011),0.899 (± 0.016),0.818 (± 0.012),Transformer,gdsc2
